# 02b — Preprocesado completo: White Balance + CLAHE + Denoising

Visualizamos el efecto de cada paso del preprocesado y del pipeline completo. Sirve para:
1. Verificar que cada técnica hace lo que esperamos.
2. Defender en la memoria por qué se usa cada una.
3. Detectar imágenes problemáticas.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import (
    preprocess,
    gray_world_white_balance,
    apply_clahe,
    bilateral_denoise,
)
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/klasson_flat')
plt.rcParams['figure.dpi'] = 90

## 1. Pipeline paso a paso sobre una imagen

Mostramos: original → +WB → +CLAHE → +Denoise (pipeline completo).

In [ ]:
category = 'Apple'
img = load_image(list_images(DATA_ROOT / category)[0])

step1 = gray_world_white_balance(img)
step2 = apply_clahe(step1)
step3 = bilateral_denoise(step2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(axes,
                          [img, step1, step2, step3],
                          ['Original', '+ White Balance', '+ CLAHE', '+ Denoise (final)']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Aislar el efecto de cada técnica

Cada paso por separado, partiendo de la imagen original.

In [ ]:
only_wb = preprocess(img, apply_wb=True, apply_clahe_step=False, apply_denoise=False)
only_clahe = preprocess(img, apply_wb=False, apply_clahe_step=True, apply_denoise=False)
only_denoise = preprocess(img, apply_wb=False, apply_clahe_step=False, apply_denoise=True)
full = preprocess(img)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, im, title in zip(axes,
                          [img, only_wb, only_clahe, only_denoise, full],
                          ['Original', 'Solo WB', 'Solo CLAHE', 'Solo Denoise', 'Pipeline completo']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Galería sobre varias categorías

In [ ]:
categories = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])[:6]
fig, axes = plt.subplots(len(categories), 2, figsize=(8, 3 * len(categories)))
for i, cat in enumerate(categories):
    images = list_images(DATA_ROOT / cat)
    if not images:
        continue
    img = load_image(images[0])
    img_pre = preprocess(img)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f'{cat} — original', fontsize=10); axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pre); axes[i, 1].set_title(f'{cat} — WB+CLAHE+Denoise', fontsize=10); axes[i, 1].axis('off')
plt.tight_layout(); plt.show()

## 4. Conclusiones para la memoria

- **White Balance** corrige el tinte de iluminación (notable en imágenes con focos cálidos).
- **CLAHE** mejora el contraste local en zonas oscuras y claras.
- **Bilateral Denoise** elimina ruido de cámara conservando los bordes de los productos.